# Advanced Model Quantization & Compression

**Topics Covered:**
- Quantization fundamentals (FP32/FP16/BF16/FP8/INT8/INT4/NF4)
- Post-Training Quantization (GPTQ, AWQ, SmoothQuant, GGUF, SpQR, AQLM, QuIP#, HQQ, EETQ)
- Quantization-Aware Training (QAT, fake quantization, STE, QLoRA)
- Pruning (unstructured, structured, Lottery Ticket, SparseGPT, Wanda, LLM-Pruner)
- Knowledge Distillation (response-based, feature-based, relation-based, DistilBERT, TinyBERT)
- Neural Architecture Search (DARTS, ENAS, Once-for-All, BigNAS)
- Efficient Inference (ONNX, TorchScript, torch.compile, OpenVINO, TensorRT)

## 1. Quantization Fundamentals

Quantization maps high-precision floating-point values to lower-precision representations, reducing memory footprint and accelerating inference.

### Numeric Formats

| Format | Bits | Sign | Exponent | Mantissa | Range | Use Case |
|--------|------|------|----------|----------|-------|----------|
| FP32   | 32   | 1    | 8        | 23       | ±3.4e38 | Training default |
| FP16   | 16   | 1    | 5        | 10       | ±65504  | Mixed precision |
| BF16   | 16   | 1    | 8        | 7        | ±3.4e38 | TPU/Ampere+ |
| FP8 E4M3 | 8  | 1    | 4        | 3        | ±448    | Forward pass |
| FP8 E5M2 | 8  | 1    | 5        | 2        | ±57344  | Gradient |
| INT8   | 8    | 1    | 0        | 7        | -128..127 | Inference |
| INT4   | 4    | 1    | 0        | 3        | -8..7   | LLM weights |
| NF4    | 4    |    |        |        | normal distr. | QLoRA |

### Memory Savings vs FP32

```
FP32  ████████████████  100%  (4 bytes/param)
BF16  ████████          50%   (2 bytes/param)
FP16  ████████          50%   (2 bytes/param)
INT8  ████              25%   (1 byte/param)
INT4  ██                12.5% (0.5 bytes/param)
INT2  █                 6.25% (0.25 bytes/param)
```

A 7B parameter model: FP32 = 28 GB, BF16 = 14 GB, INT4 = 3.5 GB

### Quantization Math

**Uniform Affine Quantization (asymmetric):**

$$x_q = \text{clamp}\left(\text{round}\left(\frac{x}{s}\right) + z,\; q_{\min},\; q_{\max}\right)$$

$$x \approx s \cdot (x_q - z)$$

where $s = \frac{x_{\max} - x_{\min}}{q_{\max} - q_{\min}}$ is the scale and $z = q_{\min} - \frac{x_{\min}}{s}$ is the zero point.

**Symmetric quantization** ($z = 0$):
$$s = \frac{\max(|x|)}{2^{b-1} - 1}$$

**Quantization Error (SQNR):**
$$\text{SQNR} = 10 \log_{10}\left(\frac{\sigma_x^2}{\sigma_e^2}\right) \approx 6.02b + 1.76 \text{ dB}$$

Each additional bit yields ~6 dB improvement.

**Per-tensor vs Per-channel:** Per-channel uses one scale per output channel, significantly reducing error for weight tensors with varying ranges across channels.

### PTQ vs QAT Tradeoffs

```
PTQ (Post-Training Quantization)
  Pretrained FP32 model
       │
       ▼
  Calibration data (128-512 samples)
       │
       ▼
  Compute scales/zero-points
       │
       ▼
  Quantized model ──► Fast, no retraining, slight accuracy loss

QAT (Quantization-Aware Training)
  Pretrained FP32 model
       │
       ▼
  Insert fake-quant nodes in graph
       │
       ▼
  Fine-tune with STE gradients (hours-days)
       │
       ▼
  Quantized model ──► Best accuracy, needs training resources
```

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Simulate quantization of a weight tensor
np.random.seed(42)
weights = np.random.randn(1000).astype(np.float32)

def quantize_symmetric(x, bits):
    n_levels = 2 ** (bits - 1) - 1
    scale = np.max(np.abs(x)) / n_levels
    x_q = np.clip(np.round(x / scale), -n_levels, n_levels)
    x_dq = x_q * scale
    mse = np.mean((x - x_dq) ** 2)
    snr = 10 * np.log10(np.var(x) / (mse + 1e-12))
    return x_dq, scale, mse, snr

print('Quantization Error vs Bit-width (symmetric, per-tensor):')
print(f'{"Bits":>6} {"Scale":>10} {"MSE":>12} {"SNR (dB)":>10}')
for bits in [8, 4, 3, 2]:
    _, scale, mse, snr = quantize_symmetric(weights, bits)
    print(f'INT{bits}  {scale:>10.5f} {mse:>12.6f} {snr:>10.2f}')

# NF4 quantile levels
from scipy.stats import norm
k = 4  # 4-bit
n_bins = 2**k
# NF4 levels are midpoints of equal-probability bins on N(0,1)
probs = np.linspace(0, 1, n_bins + 1)[1:-1]
quantiles = norm.ppf(probs)
nf4_levels = (quantiles[:-1] + quantiles[1:]) / 2
# Add symmetric endpoints
nf4_levels = np.concatenate([[-np.abs(nf4_levels).max()], nf4_levels, [np.abs(nf4_levels).max()]])
nf4_levels = nf4_levels / nf4_levels.max()  # normalize to [-1, 1]

print(f'\nNF4 quantization levels (normalized): {nf4_levels.round(3)}')
print('NF4 uses non-uniform levels denser near 0 where Gaussian weight mass is highest')

Quantization Error vs Bit-width (symmetric, per-tensor):
  Bits      Scale          MSE   SNR (dB)
INT8     0.03034     0.000076      40.98
INT4     0.55039     0.025197      15.80
INT3     1.28424     0.142653       8.27
INT2     3.85273     0.819485       0.68



NF4 quantization levels (normalized): [-1.    -1.    -0.759 -0.582 -0.433 -0.301 -0.177 -0.059  0.059  0.177
  0.301  0.433  0.582  0.759  1.     1.   ]
NF4 uses non-uniform levels denser near 0 where Gaussian weight mass is highest


### BF16 vs FP16: Why BF16 Wins for Training

```
FP32:  S EEEEEEEE MMMMMMMMMMMMMMMMMMMMMMM   (1+8+23 bits)
FP16:  S EEEEE MMMMMMMMMM                   (1+5+10 bits)
BF16:  S EEEEEEEE MMMMMMM                   (1+8+7  bits)
```

**BF16 = truncated FP32**: same exponent range as FP32, fewer mantissa bits.
- No overflow during training (critical for gradient accumulation)
- Lossless cast to/from FP32
- FP16 can overflow at values > 65504 (common in gradient norms)

### NF4 (NormalFloat4) Used in QLoRA

NF4 is **information-theoretically optimal** 4-bit quantization for normally distributed data.
Instead of uniform levels, uses the quantiles of $\mathcal{N}(0,1)$:

$$q_i = \frac{1}{2}\left(Q^{-1}\left(\frac{i}{2^k+1}\right) + Q^{-1}\left(\frac{i+1}{2^k+1}\right)\right)$$

Ensures **equal probability mass** between levels → minimizes expected quantization error for Gaussian weights.

**Double Quantization** (from QLoRA): quantize the quantization constants (scales/zeros) themselves from FP32 to FP8, saving ~0.5 bits/param additional.

## 2. Post-Training Quantization (PTQ) Methods

### Taxonomy

```
PTQ Methods
├── Weight-only quantization (activations stay FP16)
│   ├── GPTQ    layer-wise OBS, column-by-column
│   ├── AWQ     activation-aware scaling before quant
│   ├── HQQ     half-quadratic, no calibration data
│   ├── AQLM    additive vector quantization (2-bit)
│   └── QuIP#   incoherence + E8 lattice codebook (2-bit)
├── Weight + Activation quantization
│   ├── SmoothQuant migrate outliers from act to weight
│   ├── FP8         H100 native W8A8
│   └── EETQ        easy & efficient INT8
└── Mixed-format storage
    ├── GGUF/GGML llama.cpp CPU/GPU format
    └── SpQR      sparse outliers in FP16, rest INT4
```

### Comparison Table

| Method | Bits | Accuracy | GPU Req | Calibration | Notes |
|--------|------|----------|---------|-------------|-------|
| GPTQ   | 2-8  | Good     | Yes     | Yes (~128 samples) | Layer-wise OBS |
| AWQ    | 4    | Better   | Yes     | Yes | Salient weight protection |
| SmoothQuant | 8 | Excellent | Yes  | Yes | W8A8 |
| GGUF Q4_K_M | 4.5 | Good  | Optional | No | CPU-friendly |
| SpQR   | ~4.1 | Excellent | Yes    | Yes | Outliers in FP16 |
| AQLM   | 2    | Good     | Yes     | Yes | Additive codebook |
| QuIP#  | 2    | Best 2-bit | Yes   | Yes | Lattice E8 codebook |
| HQQ    | 2-8  | Good     | Yes     | **No** | Half-quadratic opt |
| EETQ   | 8    | Excellent | Yes    | No  | INT8 act per-token |
| FP8    | 8    | Best     | H100    | Yes | Hardware native |

### GPTQ Algorithm (Frantar et al., 2022)

Based on **Optimal Brain Surgeon (OBS)**. For each transformer layer, minimize:

$$\min_{\hat{W}} \| WX - \hat{W}X \|_F^2$$

The Hessian $H = 2XX^T$ governs sensitivity. When quantizing column $j$:

$$\delta W_{:,j} = -\frac{W_{:,j} - Q(W_{:,j})}{[H^{-1}]_{jj}} \cdot H^{-1}_{:,j}$$

Key tricks: Cholesky-based $H^{-1}$ update, activation ordering (`desc_act=True`), group-wise quantization.

**Achieves INT4 with < 1% perplexity increase** on LLaMA-2-7B.

### AWQ Algorithm (Lin et al., 2023)

Key insight: **1% of weight channels (salient) dominate quantization error**.
Identify salient channels by activation magnitude $s_j = \max|X_j|$, then scale:

$$Y = X W^T = (X \cdot \text{diag}(s)^{-1})(\text{diag}(s) \cdot W^T) = \hat{X}\hat{W}^T$$

Scales $s_j = \max(|X_j|)^\alpha / \max(|W_j|)^{1-\alpha}$ chosen to minimize INT4 quantization error.
**No weight updates** just smooth scaling absorbed into adjacent layers.

In [2]:
# GPTQ Quantization Example
# pip install auto-gptq transformers accelerate

GPTQ_EXAMPLE = '''
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig
from transformers import AutoTokenizer

model_name = "facebook/opt-125m"  # or any HF causal LM
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Configure quantization
quantize_config = BaseQuantizeConfig(
    bits=4,                  # 4-bit quantization
    group_size=128,          # per-group quantization (better accuracy)
    desc_act=True,           # activation ordering (Frantar et al.)
    sym=False,               # asymmetric quantization
    true_sequential=True,    # quantize in forward order
)

# Load model for quantization
model = AutoGPTQForCausalLM.from_pretrained(
    model_name,
    quantize_config=quantize_config
)

# Prepare calibration data
calibration_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Quantization reduces model size while preserving accuracy.",
    "Large language models are transforming AI applications.",
    # Aim for 128+ diverse examples from target domain
]
examples = [
    tokenizer(text, return_tensors="pt", max_length=512, truncation=True)
    for text in calibration_texts
]

# Run GPTQ quantization (5-30 min depending on model size)
model.quantize(examples)

# Save quantized model
model.save_quantized("./opt-125m-4bit-gptq", use_safetensors=True)
tokenizer.save_pretrained("./opt-125m-4bit-gptq")

# Load and use quantized model for inference
model_q = AutoGPTQForCausalLM.from_quantized(
    "./opt-125m-4bit-gptq",
    device="cuda:0",
    use_safetensors=True,
    inject_fused_attention=True,   # faster attention
    inject_fused_mlp=True,         # faster MLP
)

# Generate text
from transformers import pipeline
pipe = pipeline("text-generation", model=model_q, tokenizer=tokenizer)
print(pipe("Once upon a time", max_new_tokens=50)[0]["generated_text"])
'''

print('GPTQ Quantization Workflow:')
print(GPTQ_EXAMPLE)

# Demonstrate GPTQ-style group quantization
import torch

def gptq_quantize_weight_demo(W, bits=4, group_size=128):
    """Simplified GPTQ-style weight quantization (without Hessian compensation)."""
    W = W.float()
    n_levels = 2 ** (bits - 1) - 1
    rows, cols = W.shape
    W_q = torch.zeros_like(W)
    scales = []

    for start in range(0, cols, group_size):
        end = min(start + group_size, cols)
        group = W[:, start:end]
        scale = group.abs().max() / n_levels
        if scale > 0:
            q = torch.clamp(torch.round(group / scale), -n_levels, n_levels)
            W_q[:, start:end] = q * scale
        scales.append(scale.item())

    mse = ((W - W_q) ** 2).mean().item()
    return W_q, scales, mse

torch.manual_seed(42)
W_sample = torch.randn(64, 256)
_, _, mse4 = gptq_quantize_weight_demo(W_sample, bits=4)
_, _, mse8 = gptq_quantize_weight_demo(W_sample, bits=8)
_, _, mse4g = gptq_quantize_weight_demo(W_sample, bits=4, group_size=32)

print(f'Weight matrix shape: {W_sample.shape}')
print(f'INT8 (group=128): MSE = {mse8:.6f}')
print(f'INT4 (group=128): MSE = {mse4:.6f}')
print(f'INT4 (group=32):  MSE = {mse4g:.6f} (smaller group = better accuracy)')
print(f'\nMemory breakdown:')
print(f'  FP32: {W_sample.numel() * 4} bytes')
print(f'  INT4 (group=128): {W_sample.numel() * 0.5:.0f} bytes weights + {256//128 * 64 * 4:.0f} bytes scales')

GPTQ Quantization Workflow:

from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig
from transformers import AutoTokenizer

model_name = "facebook/opt-125m"  # or any HF causal LM
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Configure quantization
quantize_config = BaseQuantizeConfig(
    bits=4,                  # 4-bit quantization
    group_size=128,          # per-group quantization (better accuracy)
    desc_act=True,           # activation ordering (Frantar et al.)
    sym=False,               # asymmetric quantization
    true_sequential=True,    # quantize in forward order
)

# Load model for quantization
model = AutoGPTQForCausalLM.from_pretrained(
    model_name,
    quantize_config=quantize_config
)

# Prepare calibration data
calibration_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Quantization reduces model size while preserving accuracy.",
    "Large language models are transforming AI applications.",
    # Aim for 128+ diverse exa

Weight matrix shape: torch.Size([64, 256])
INT8 (group=128): MSE = 0.000082
INT4 (group=128): MSE = 0.027298
INT4 (group=32):  MSE = 0.023244 (smaller group = better accuracy)

Memory breakdown:
  FP32: 65536 bytes
  INT4 (group=128): 8192 bytes weights + 512 bytes scales


In [3]:
# AWQ (Activation-aware Weight Quantization) Example
# pip install autoawq

AWQ_EXAMPLE = '''
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

model_path = "mistralai/Mistral-7B-v0.1"
quant_path  = "mistral-7b-awq-4bit"

quant_config = {
    "zero_point": True,   # asymmetric quantization
    "q_group_size": 128,  # group size
    "w_bit": 4,           # 4-bit weights
    "version": "GEMM"     # GEMM (batch) | GEMV (single token)
}

# Load model
model = AutoAWQForCausalLM.from_pretrained(model_path, safetensors=True)
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

# Quantize using AWQ (auto downloads calibration dataset)
model.quantize(tokenizer, quant_config=quant_config)

# Save quantized model
model.save_quantized(quant_path, safetensors=True)
tokenizer.save_pretrained(quant_path)

# Load quantized for inference (fuse_layers=True for speed)
model_q = AutoAWQForCausalLM.from_quantized(quant_path, fuse_layers=True)

# Also works via transformers:
# from transformers import AutoModelForCausalLM
# model = AutoModelForCausalLM.from_pretrained(quant_path)
'''

print('AWQ Quantization Workflow:')
print(AWQ_EXAMPLE)

# Demonstrate AWQ channel scaling principle
import torch

def awq_find_scale(weight, activation_stats, alpha=0.5):
    """
    AWQ scale: balance quant difficulty between W and X.
    scale = max|x_c|^alpha / max|w_c|^(1-alpha)
    """
    w_max = weight.abs().max(dim=0).values   # per input-channel
    x_max = activation_stats                  # per input-channel
    scale = (x_max.pow(alpha) / (w_max.pow(1 - alpha) + 1e-8)).clamp(min=1e-8)
    scale = scale / scale.mean()  # normalize
    W_scaled = weight * scale.unsqueeze(0)
    return W_scaled, scale

torch.manual_seed(0)
W = torch.randn(64, 128)
x_stats = torch.abs(torch.randn(128)) + 0.5
# Introduce outliers (common in LLMs channels 10, 42 dominate)
x_stats[10] = 15.0
x_stats[42] = 12.0

W_scaled, scale = awq_find_scale(W, x_stats)

print('AWQ Channel Scaling Demo:')
print(f'Original W range:   [{W.min():.3f}, {W.max():.3f}]')
print(f'Scaled   W range:   [{W_scaled.min():.3f}, {W_scaled.max():.3f}]')
print(f'Outlier channel 10: x_stat={x_stats[10]:.1f}, scale={scale[10]:.3f}')
print(f'Outlier channel 42: x_stat={x_stats[42]:.1f}, scale={scale[42]:.3f}')
print(f'Normal channel 0:   x_stat={x_stats[0]:.3f}, scale={scale[0]:.3f}')
print('\nHigher-activation channels get larger weight scale')
print('=> Weights "absorb" more precision for important channels')

AWQ Quantization Workflow:

from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

model_path = "mistralai/Mistral-7B-v0.1"
quant_path  = "mistral-7b-awq-4bit"

quant_config = {
    "zero_point": True,   # asymmetric quantization
    "q_group_size": 128,  # group size
    "w_bit": 4,           # 4-bit weights
    "version": "GEMM"     # GEMM (batch) | GEMV (single token)
}

# Load model
model = AutoAWQForCausalLM.from_pretrained(model_path, safetensors=True)
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

# Quantize using AWQ (auto downloads calibration dataset)
model.quantize(tokenizer, quant_config=quant_config)

# Save quantized model
model.save_quantized(quant_path, safetensors=True)
tokenizer.save_pretrained(quant_path)

# Load quantized for inference (fuse_layers=True for speed)
model_q = AutoAWQForCausalLM.from_quantized(quant_path, fuse_layers=True)

# Also works via transformers:
# from transformers import AutoModelForCausalLM

### SmoothQuant (Xiao et al., 2022)

LLMs have **activation outliers** (~100× larger than typical) concentrated in fixed channels.
These break INT8 quantization. SmoothQuant migrates difficulty from activations to weights:

$$Y = (X \cdot \text{diag}(s)^{-1}) \cdot (\text{diag}(s) \cdot W) = \hat{X} \cdot \hat{W}$$

$$s_j = \frac{\max(|X_j|)^\alpha}{\max(|W_j|)^{1-\alpha}}, \quad \alpha = 0.5 \text{ (default)}$$

Achieves **W8A8** with near-lossless accuracy. The scale vector is absorbed offline zero overhead.

### GGUF / GGML (llama.cpp)

GGUF (GPT-Generated Unified Format) is a binary file format for CPU/GPU inference:

```
GGUF File Layout:
┌──────────────────────────────┐
│ Header: magic "GGUF" + version│
│ n_tensors, n_kv              │
├──────────────────────────────┤
│ Metadata KV pairs:           │
│  general.architecture        │
│  llama.context_length        │
│  tokenizer.ggml.model        │
│  tokenizer.ggml.tokens       │
├──────────────────────────────┤
│ Tensor Info Array:           │
│  [name, shape, type, offset] │
├──────────────────────────────┤
│ Tensor Data (quantized):     │
│  Q2_K, Q3_K, Q4_K, Q5_K,    │
│  Q6_K, Q8_0, F16, F32       │
└──────────────────────────────┘

Q4_K_M: 4-bit weights, 6-bit scales/mins
        ~4.85 bits/param effective, great quality/speed tradeoff
```

### SpQR (Sparse-Quantized Representation)

Identifies ~**1% of weights as sensitive outliers** and stores them in FP16:
- ~1% outlier weights: FP16 (2 bytes each)
- ~99% normal weights: INT4 (0.5 bytes each)
- Net: ~4.1 bits/param with near-lossless quality on GPT-family models

### HQQ (Half-Quadratic Quantization, Badri & Shaji, 2023)

Formulates quantization as optimization **no calibration data needed**:

$$\min_{W_q, s, z} \| W - s(W_q - z) \|^2_F + \lambda \| W_q - W_q^{\text{ref}} \|_1$$

Solved via **half-quadratic splitting** in seconds. Supports 1-8 bit with group quantization.

### AQLM (Additive Quantization, Egiazarian et al., 2024)

Extends vector quantization to 2-bit using **multiple additive codebooks**:

$$W_j \approx \sum_{m=1}^{M} C_m[I_{j,m}]$$

where $C_m \in \mathbb{R}^{k \times d}$ are codebooks and $I_{j,m}$ are indices. With $M=2, k=256$: effectively 2 bits/param.

### QuIP# (Tseng et al., 2024)

Makes weights **incoherent** before quantization using random Hadamard transforms:
1. Apply random orthogonal transform: $\tilde{W} = Q W Q^T$
2. Quantize $\tilde{W}$ with the E8 lattice codebook (optimal 8D sphere packing)
3. At runtime: apply $Q^T$ and $Q$ (fast with WHT $O(n \log n)$)

**Best 2-bit quality** on the Llama family as of 2024.

## 3. Quantization-Aware Training (QAT)

QAT simulates quantization during training using **fake quantization** nodes, allowing gradients to flow through the discretization step.

### Fake Quantization

```
Forward pass:  x ──► round(x/s)·s  (quantize then immediately dequantize)
                      └─ values are quantized, dtype stays FP32

Backward pass: gradient passes through unchanged (STE approximation)

Computational graph:
   FP32 weights ──► fake_quant(·) ──► FP32 (quantized values) ──► loss
        ▲               ↑STE                                          │
        └──────── ∂loss/∂w ≈ ∂loss/∂x_q ◄──────────────────────────┘
```

### Straight-Through Estimator (STE, Bengio et al., 2013)

The round function has zero gradient almost everywhere. STE approximates the gradient:

$$\frac{\partial \mathcal{L}}{\partial x} \approx \frac{\partial \mathcal{L}}{\partial x_q} \cdot \mathbf{1}[q_{\min} \le x \le q_{\max}]$$

Gradients flow as if quantization didn't happen (within the clipping range). Clipping prevents gradients for out-of-range values, encouraging the model to keep weights within the quantization range.

### Learnable Quantization Parameters

PACT, LSQ, and LSQ+ make scale $s$ and zero-point $z$ trainable:

$$\frac{\partial \mathcal{L}}{\partial s} = \frac{\partial \mathcal{L}}{\partial x_q} \cdot \frac{\partial x_q}{\partial s}$$

This allows the model to learn optimal quantization ranges per layer.

### QLoRA (Dettmers et al., 2023)

QLoRA = **NF4-quantized base model** + **BF16 LoRA adapters**:

```
Input x (BF16)
   │
   ├──► Dequantize(W_NF4) → W_BF16  (frozen base weights)
   │         │
   │    W_BF16 · x  (base computation)
   │
   └──► LoRA: A·(B·x)  (trainable BF16 adapters, rank r)
         │
         ▼
   Output = W_BF16·x + A·B·x
```

Key innovations:
1. **NF4 quantization** for normally distributed base weights
2. **Double quantization**: quantize quantization constants (FP32 → FP8)
3. **Paged optimizers**: NVLink CPU offload for optimizer states
4. Memory savings: 65B model fits on a single 48GB GPU for fine-tuning

**Memory formula** for QLoRA:
$$M = N \times 0.5 \text{ (NF4)} + 2Nr/d \times 2 \text{ (LoRA BF16)}$$
where $N$ = params, $r$ = LoRA rank, $d$ = hidden dim.

In [4]:
import torch
import torch.nn as nn
import torch.quantization

# === QAT with PyTorch ===

class QATModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.quant  = torch.quantization.QuantStub()   # mark input quant point
        self.fc1    = nn.Linear(128, 64)
        self.relu   = nn.ReLU()
        self.fc2    = nn.Linear(64, 10)
        self.dequant = torch.quantization.DeQuantStub()  # mark output dequant

    def forward(self, x):
        x = self.quant(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        x = self.dequant(x)
        return x

model = QATModel()

# QAT config: fbgemm for x86, qnnpack for ARM
model.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')

# prepare_qat inserts FakeQuantize modules before every quantizable op
model_qat = torch.quantization.prepare_qat(model.train())
print('QAT model structure (note FakeQuantize nodes):')
print(model_qat)

# Training loop (simulated)
optimizer = torch.optim.AdamW(model_qat.parameters(), lr=1e-3)
x_dummy = torch.randn(32, 128)
y_dummy = torch.randint(0, 10, (32,))

model_qat.train()
print('\nQAT training (simulated):')
for step in range(5):
    logits = model_qat(x_dummy)
    loss = nn.CrossEntropyLoss()(logits, y_dummy)
    optimizer.zero_grad()
    loss.backward()   # STE gradient flows through fake-quant nodes
    optimizer.step()
    print(f'  Step {step+1}: loss={loss.item():.4f}')

# Convert to real INT8: replaces FakeQuantize with quantized ops
model_qat.eval()
model_int8 = torch.quantization.convert(model_qat)
print('\nFinal INT8 model:')
print(model_int8)

# Verify it runs
with torch.no_grad():
    out = model_int8(x_dummy)
print(f'\nINT8 output shape: {out.shape}')

# === QLoRA pattern demo ===
print('\n--- QLoRA Pattern Demo ---')

class LoRALinear(nn.Module):
    """Linear layer with LoRA adapters (simulates QLoRA pattern)."""
    def __init__(self, in_f, out_f, rank=8, alpha=16):
        super().__init__()
        self.base_weight = nn.Parameter(torch.randn(out_f, in_f), requires_grad=False)
        self.lora_A = nn.Parameter(torch.randn(rank, in_f) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_f, rank))
        self.scaling = alpha / rank

    def forward(self, x):
        # Base: frozen (would be NF4 dequantized in real QLoRA)
        base_out = x @ self.base_weight.T
        # LoRA: trainable low-rank update
        lora_out = x @ self.lora_A.T @ self.lora_B.T
        return base_out + self.scaling * lora_out

lora_layer = LoRALinear(256, 256, rank=8)
trainable = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)
total     = sum(p.numel() for p in lora_layer.parameters())
print(f'Total params:     {total:,}')
print(f'Trainable params: {trainable:,} ({100*trainable/total:.1f}%)')
print(f'Base weight: frozen (simulating NF4 stored)')

QAT model structure (note FakeQuantize nodes):
QATModel(
  (quant): QuantStub(
    (activation_post_process): FusedMovingAvgObsFakeQuantize(
      fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.quint8, quant_min=0, quant_max=127, qscheme=torch.per_tensor_affine, reduce_range=True
      (activation_post_process): MovingAverageMinMaxObserver(min_val=inf, max_val=-inf)
    )
  )
  (fc1): Linear(
    in_features=128, out_features=64, bias=True
    (weight_fake_quant): FusedMovingAvgObsFakeQuantize(
      fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.qint8, quant_min=-128, quant_max=127, qscheme=torch.per_channel_symmetric, reduce_range=False
      (activation_post_process): MovingAveragePerChannelMinMaxObserver(min_val=tensor([]), max_val=tensor([]))
    )
    (activation_post_process): FusedMovingAvgObsFakeQuan

/tmp/ipykernel_69492/2061355990.py:29: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_qat = torch.quantization.prepare_qat(model.train())
/home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/.venv/lib/python3.12/site-packages/torch/ao/quantization/observer.py:534: UserWarning: Please use quant_min and quant_max to specify the range for observers.      


QAT training (simulated):
  Step 1: loss=2.2934
  Step 2: loss=2.2310
  Step 3: loss=2.1709
  Step 4: loss=2.1100
  Step 5: loss=2.0517

Final INT8 model:
QATModel(
  (quant): Quantize(scale=tensor([0.0501]), zero_point=tensor([63]), dtype=torch.quint8)
  (fc1): QuantizedLinear(in_features=128, out_features=64, scale=0.033592164516448975, zero_point=62, qscheme=torch.per_channel_affine)
  (relu): ReLU()
  (fc2): QuantizedLinear(in_features=64, out_features=10, scale=0.012447066605091095, zero_point=55, qscheme=torch.per_channel_affine)
  (dequant): DeQuantize()
)

INT8 output shape: torch.Size([32, 10])

--- QLoRA Pattern Demo ---
Total params:     69,632
Trainable params: 4,096 (5.9%)
Base weight: frozen (simulating NF4 stored)


/tmp/ipykernel_69492/2061355990.py:50: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = torch.quantization.convert(model_qat)


## 4. Neural Network Pruning

Pruning removes redundant weights/neurons to reduce model size and FLOPs.

### Pruning Taxonomy

```
Pruning
├── Unstructured (weight-level, irregular sparsity)
│   ├── Magnitude pruning    remove |w| < threshold
│   ├── Taylor expansion     |w · ∂L/∂w| importance
│   ├── OBD/OBS             Hessian-based (optimal)
│   ├── SparseGPT           OBS for LLMs in one shot
│   └── Wanda               |w| × ||x||₂ importance
└── Structured (tensor-level, hardware-friendly)
    ├── Filter/channel pruning  remove CNN output channels
    ├── Attention head pruning  remove transformer heads
    ├── Layer pruning           remove entire layers
    ├── LLM-Pruner             coupled-structure removal
    └── SliceGPT               PCA-based row/col slicing
```

### Unstructured vs Structured

| Aspect | Unstructured | Structured |
|--------|-------------|------------|
| Sparsity pattern | Arbitrary zeros | Regular tensor shapes |
| Max sparsity | 90-99% | 50-80% typically |
| Speedup on dense HW | Requires sparse BLAS | Yes (smaller tensors) |
| Accuracy at 50% | ~0-1% drop | ~1-5% drop |
| LLM examples | SparseGPT, Wanda | LLM-Pruner, SliceGPT |

### Lottery Ticket Hypothesis (Frankle & Carlin, ICLR 2019)

> *"A randomly initialized, dense network contains a subnetwork (winning ticket) that, when trained in isolation from the same initial weights, can match or exceed the full network's accuracy."*

**Iterative Magnitude Pruning (IMP):**
1. Train $f(x; \theta)$ to convergence, get $\theta^*$
2. Prune $p\%$ of weights with smallest $|\theta^*|$, creating mask $m$
3. Reset surviving weights: $\theta \leftarrow m \odot \theta_0$ (original init!)
4. Repeat from step 1 with $f(x; m \odot \theta)$

The **winning ticket** = mask $m$ + original init $\theta_0$.

**Limitation**: requires full training to find expensive for large networks.

### SparseGPT (Frantar & Alistarh, 2023)

Prunes **50-60% of LLM weights in one shot** without retraining:

For each weight column $j$, solve:
$$\min_{\delta W_F,\, W_P=0} \| W_F X - (W_F + \delta W_F)X \|_2^2$$

Uses the same Hessian inverse as GPTQ with sparse mask applied simultaneously.
Can achieve **2:4 sparsity** (NVIDIA sparse tensor core pattern) for hardware speedup.

### Wanda (Sun et al., 2023)

Extremely simple but effective pruning score:

$$S_{ij} = |W_{ij}| \cdot \|X_j\|_2$$

Prune weights with lowest $S_{ij}$. No weight updates, no Hessian required.
Comparable to SparseGPT at 50% sparsity in much less time.

### LLM-Pruner (Ma et al., 2023)

**Structured** pruning via gradient-based importance:
1. Estimate group importance: $I_g = \|g \odot \theta\|$ (gradient × weight)
2. Identify coupled groups (attention heads + FFN neurons)
3. Remove lowest-importance groups
4. LoRA-based recovery fine-tuning (3-5 hours on single GPU)

In [5]:
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune

# === Pruning with torch.nn.utils.prune ===

class PrunableNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.fc1   = nn.Linear(64 * 7 * 7, 256)
        self.fc2   = nn.Linear(256, 10)
        self.relu  = nn.ReLU()
        self.pool  = nn.MaxPool2d(2)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

def sparsity(model):
    total, zeros = 0, 0
    for m in model.modules():
        if hasattr(m, 'weight') and m.weight is not None:
            w = m.weight.data
            total += w.numel()
            zeros += (w == 0).sum().item()
    return zeros / total if total > 0 else 0

model = PrunableNet()
print(f'Before pruning: {100*sparsity(model):.1f}% sparse')

# --- 1. Global unstructured magnitude pruning ---
params_to_prune = [
    (model.conv1, 'weight'),
    (model.conv2, 'weight'),
    (model.fc1,   'weight'),
    (model.fc2,   'weight'),
]
prune.global_unstructured(
    params_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.5,
)
print(f'After 50% global L1 unstructured: {100*sparsity(model):.1f}% sparse')

# --- 2. Structured filter pruning ---
model2 = PrunableNet()
# Remove 25% of conv1 filters (output channels) by L2 norm
prune.ln_structured(model2.conv1, name='weight', amount=0.25, n=2, dim=0)
mask = model2.conv1.weight_mask
zero_filters = (mask.sum(dim=(1,2,3)) == 0).sum().item()
print(f'\nStructured filter pruning: {zero_filters}/32 conv1 filters fully zeroed')

# --- 3. Random unstructured (baseline) ---
model3 = PrunableNet()
prune.random_unstructured(model3.fc1, name='weight', amount=0.7)
fc1_sparse = (model3.fc1.weight == 0).float().mean().item()
print(f'Random 70% unstructured fc1: {100*fc1_sparse:.1f}% sparse')

# --- 4. Make pruning permanent ---
for module, name in params_to_prune:
    prune.remove(module, name)   # removes weight_mask, weight_orig buffers
print(f'After making permanent: {100*sparsity(model):.1f}% sparse (same values)')

# --- 5. Iterative Magnitude Pruning (Lottery Ticket style) ---
print('\n--- Iterative Magnitude Pruning (Lottery Ticket) ---')
model_lth = PrunableNet()
init_weights = {n: p.clone() for n, p in model_lth.named_parameters()}

layers = [(model_lth.fc1, 'weight'), (model_lth.fc2, 'weight')]
for round_num in range(4):
    # Prune 20% each round (cumulative via global)
    prune.global_unstructured(layers, pruning_method=prune.L1Unstructured, amount=0.2)
    # Reset to original initialization (the winning ticket)
    with torch.no_grad():
        for name, param in model_lth.named_parameters():
            if '_orig' in name:
                base = name.replace('_orig', '')
                if base in init_weights:
                    param.copy_(init_weights[base])
    sp = sparsity(model_lth)
    print(f'  Round {round_num+1}: {100*sp:.1f}% global sparse (weights reset to init)')

# --- 6. Wanda-style importance score ---
print('\n--- Wanda Importance Scores ---')
W = torch.randn(64, 128)
X = torch.randn(32, 128)  # batch of activations
x_norm = X.norm(dim=0)    # ||X_j||_2 across batch
wanda_scores = W.abs() * x_norm.unsqueeze(0)
# Prune 50% by Wanda score
threshold = wanda_scores.flatten().kthvalue(int(0.5 * wanda_scores.numel())).values
mask = (wanda_scores > threshold).float()
print(f'Wanda sparsity: {100*(1-mask.mean().item()):.1f}%')
print(f'Wanda preserves weights with high |w| AND high activation norm')

Before pruning: 0.0% sparse
After 50% global L1 unstructured: 50.0% sparse

Structured filter pruning: 8/32 conv1 filters fully zeroed


Random 70% unstructured fc1: 70.0% sparse


After making permanent: 50.0% sparse (same values)

--- Iterative Magnitude Pruning (Lottery Ticket) ---
  Round 1: 19.5% global sparse (weights reset to init)
  Round 2: 35.2% global sparse (weights reset to init)
  Round 3: 47.7% global sparse (weights reset to init)


  Round 4: 57.7% global sparse (weights reset to init)

--- Wanda Importance Scores ---
Wanda sparsity: 50.0%
Wanda preserves weights with high |w| AND high activation norm


## 5. Knowledge Distillation

Transfer knowledge from a large **teacher** to a smaller **student**.

### Overview

```
┌──────────────────────────────────────────────────────────┐
│                 TEACHER MODEL (large)                    │
│  Input → [Emb] → [L1] → [L2] → ... → [LN] → [Head]     │
│                     ↓      ↓              ↓     ↓        │
│                  feat₁  feat₂         featN  logits      │
└──────────────────────────────────────────────────────────┘
         ↓feature loss  ↓feature loss    ↓response loss
┌──────────────────────────────────────────────────────────┐
│                 STUDENT MODEL (small)                    │
│  Input → [Emb] → [L1] → [L2] → ... → [LM] → [Head]     │
└──────────────────────────────────────────────────────────┘
```

### 1. Response-Based KD (Hinton et al., 2015)

Student mimics teacher's **soft probability distributions** (dark knowledge):

$$\mathcal{L}_{KD} = (1-\alpha)\mathcal{L}_{CE}(y, p_S) + \alpha T^2 \cdot \text{KL}\left(\sigma\left(\frac{z_T}{T}\right) \| \sigma\left(\frac{z_S}{T}\right)\right)$$

- $T$ = temperature: higher $T$ → softer distributions → more transfer of non-argmax information
- $\alpha$ = weight between hard label loss and KD loss
- $T^2$ factor: counteracts the $1/T^2$ effect of KL divergence under temperature scaling

**Why soft labels work**: teacher outputs like [0.7, 0.2, 0.1] carry more information than one-hot [1, 0, 0]. The 0.2 on class 2 signals similarity to class 1.

### 2. Feature-Based KD (FitNets, Romero et al., 2014)

Student mimics teacher's **intermediate layer representations**:

$$\mathcal{L}_{feat} = \sum_{l \in \mathcal{L}} \| f_T^l(x) - r^l(f_S^l(x)) \|_F^2$$

where $r^l(\cdot)$ is a learnable projection layer matching student→teacher dimensions.

### 3. Relation-Based KD (RKD, Park et al., 2019)

Preserve **structural relationships** between examples in embedding space:

$$\mathcal{L}_{RKD} = \sum_{(i,j) \in \mathcal{X}^2} l\left(d_T(i,j),\; d_S(i,j)\right)$$

where $d(i,j) = \|f(x_i) - f(x_j)\|_2 / \bar{d}$ (normalized pairwise distance).

### Distilled Models Comparison

| Model | Teacher | Student Params | Teacher Params | Speedup | GLUE Score | Method |
|-------|---------|---------------|----------------|---------|------------|--------|
| DistilBERT | BERT-Base | 66M | 110M | 1.6× | 97% teacher | Response + hidden |
| TinyBERT-4L | BERT-Base | 14.5M | 110M | 9.4× | 96.8% teacher | Full layer-map |
| TinyBERT-6L | BERT-Base | 66M | 110M | 1.6× | 99.3% teacher | Full layer-map |
| MobileBERT | BERT-Large | 25.3M | 334M | 4× | 99.5% teacher | Bottleneck IB |
| BERT-PKD-6 | BERT-Base | 66M | 110M | 2× | 96.3% teacher | Patient KD |
| MiniLMv2 | RoBERTa-Large | 22M | 355M | 5× | Competitive | Self-attn relation |

### TinyBERT Architecture

Two-stage distillation with 4 loss components:

$$\mathcal{L}_{TinyBERT} = \mathcal{L}_{emb} + \sum_{k=1}^{K} \mathcal{L}_{attn}^k + \sum_{k=1}^{K} \mathcal{L}_{hidn}^k + \mathcal{L}_{pred}$$

- $\mathcal{L}_{emb}$: MSE on embedding layer
- $\mathcal{L}_{attn}^k$: MSE on attention matrices $A_k$
- $\mathcal{L}_{hidn}^k$: MSE on hidden states (with projection)
- $\mathcal{L}_{pred}$: soft cross-entropy on logits

### Speculative Decoding (Chen et al., 2023)

Uses a small **draft model** to propose tokens, verified by large model in parallel:

```
Step 1: Draft model generates K tokens cheaply:
        x_1 → [t₁, t₂, t₃, t₄, t₅]  (K=5 speculative tokens)

Step 2: Target model verifies all K+1 positions in ONE forward pass:
        P_target(x_1, t₁, t₂, t₃, t₄, t₅)

Step 3: Accept/reject with probability:
        Accept tᵢ if u ~ Uniform(0,1) ≤ P_target(tᵢ) / P_draft(tᵢ)
        Reject: resample from adjusted distribution, discard rest

Speedup: 2-4× for autoregressive generation
Quality: guaranteed identical distribution to target model
```

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

# === BERT → TinyBERT-style Knowledge Distillation ===

class TeacherBERT(nn.Module):
    """Simplified BERT-like teacher (6 layers, hidden=256)."""
    def __init__(self, vocab=1000, hidden=256, n_layers=6, heads=8, n_classes=4):
        super().__init__()
        self.embed = nn.Embedding(vocab, hidden)
        enc_layer = nn.TransformerEncoderLayer(
            hidden, heads, hidden*4, batch_first=True, norm_first=True, dropout=0.0)
        self.encoder = nn.TransformerEncoder(enc_layer, n_layers)
        self.head = nn.Linear(hidden, n_classes)
        self.hidden_size = hidden

    def forward(self, x):
        h = self.embed(x)
        h = self.encoder(h)
        cls = h[:, 0]       # CLS token pooling
        return self.head(cls), h  # (logits, hidden states)

class StudentTinyBERT(nn.Module):
    """TinyBERT-style student (4 layers, hidden=128)."""
    def __init__(self, vocab=1000, hidden=128, n_layers=4, heads=4,
                 n_classes=4, teacher_hidden=256):
        super().__init__()
        self.embed = nn.Embedding(vocab, hidden)
        enc_layer = nn.TransformerEncoderLayer(
            hidden, heads, hidden*4, batch_first=True, norm_first=True, dropout=0.0)
        self.encoder = nn.TransformerEncoder(enc_layer, n_layers)
        self.head = nn.Linear(hidden, n_classes)
        # Projection layers for feature-based distillation
        self.hidden_proj = nn.Linear(hidden, teacher_hidden)
        self.embed_proj  = nn.Linear(hidden, teacher_hidden)

    def forward(self, x):
        h = self.embed(x)
        h = self.encoder(h)
        cls = h[:, 0]
        return self.head(cls), h

def tinybert_loss(s_logits, t_logits, labels,
                   s_hidden, t_hidden,
                   s_embed, t_embed,
                   hidden_proj, embed_proj,
                   T=4.0, alpha=0.5, beta=0.1, gamma=0.05):
    """Combined TinyBERT-style loss."""
    # 1. Hard label loss
    ce = F.cross_entropy(s_logits, labels)

    # 2. Response-based KD (soft labels)
    s_soft = F.log_softmax(s_logits / T, dim=-1)
    t_soft = F.softmax(t_logits / T, dim=-1)
    kd = F.kl_div(s_soft, t_soft, reduction='batchmean') * (T ** 2)

    # 3. Hidden state distillation (CLS token only for simplicity)
    s_h = hidden_proj(s_hidden[:, 0])
    t_h = t_hidden[:, 0].detach()
    feat = F.mse_loss(s_h, t_h)

    # 4. Embedding distillation
    s_e = embed_proj(s_embed[:, 0])
    t_e = t_embed[:, 0].detach()
    emb = F.mse_loss(s_e, t_e)

    total = (1 - alpha) * ce + alpha * kd + beta * feat + gamma * emb
    return total, dict(ce=ce.item(), kd=kd.item(), feat=feat.item(), emb=emb.item())

# Setup
torch.manual_seed(42)
VOCAB, SEQ_LEN, BATCH, N_CLASSES = 1000, 32, 8, 4
teacher = TeacherBERT(VOCAB, n_classes=N_CLASSES)
student = StudentTinyBERT(VOCAB, n_classes=N_CLASSES)

t_params = sum(p.numel() for p in teacher.parameters())
s_params = sum(p.numel() for p in student.parameters())
print(f'Teacher: {t_params:,} params')
print(f'Student: {s_params:,} params  ({t_params/s_params:.1f}× compression)')

# Distillation training
optimizer = AdamW(student.parameters(), lr=5e-4, weight_decay=0.01)
teacher.eval()

print('\nTinyBERT Distillation:')
print(f'{"Step":>5} {"Total":>8} {"CE":>8} {"KD":>8} {"Feat":>8} {"Emb":>8}')
for step in range(6):
    x = torch.randint(0, VOCAB, (BATCH, SEQ_LEN))
    labels = torch.randint(0, N_CLASSES, (BATCH,))

    with torch.no_grad():
        t_logits, t_hidden = teacher(x)
        t_embed = teacher.embed(x)

    s_logits, s_hidden = student(x)
    s_embed = student.embed(x)

    loss, breakdown = tinybert_loss(
        s_logits, t_logits, labels,
        s_hidden, t_hidden,
        s_embed, t_embed,
        student.hidden_proj, student.embed_proj
    )
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
    optimizer.step()

    print(f'{step+1:>5} {loss.item():>8.4f} {breakdown["ce"]:>8.4f} '
          f'{breakdown["kd"]:>8.4f} {breakdown["feat"]:>8.4f} {breakdown["emb"]:>8.4f}')

Teacher: 4,995,588 params
Student: 987,652 params  (5.1× compression)

TinyBERT Distillation:
 Step    Total       CE       KD     Feat      Emb
    1   1.6609   1.4941   0.7195   4.8696   1.3429
    2   1.6797   1.4794   0.8928   4.2979   1.2754


/tmp/ipykernel_69492/2887702089.py:15: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, n_layers)
/tmp/ipykernel_69492/2887702089.py:33: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, n_layers)


    3   2.0629   2.5224   0.5873   4.4115   1.3372


    4   2.1512   2.4268   0.8272   4.5891   1.3060
    5   1.9044   2.1650   0.6054   4.5503   1.2842
    6   1.7592   2.0960   0.4529   4.1787   1.3386


## 6. Neural Architecture Search (NAS)

NAS automates neural network architecture design finding optimal depth, width, kernel sizes, and connection patterns.

### Problem Formulation

$$\alpha^* = \arg\min_{\alpha \in \mathcal{A}} \mathcal{L}_{val}(w^*(\alpha), \alpha)$$
$$\text{where } w^*(\alpha) = \arg\min_w \mathcal{L}_{train}(w, \alpha)$$

- $\alpha$ = architecture parameters (discrete choices)
- $w$ = network weights (continuous)
- $\mathcal{A}$ = search space (exponentially large)

### Search Space: Cell-based (DARTS)

```
Normal Cell (spatial resolution preserved):

  h_{i-2} ─┬─ op_{(0,2)} ──┐
            │               ├─→ node₂ = add
  h_{i-1} ─┼─ op_{(1,2)} ──┘
            │
  h_{i-2} ─┼─ op_{(0,3)} ──┐
            │               ├─→ node₃ = add
  node₂   ─┼─ op_{(2,3)} ──┘
            │     ...

  Candidate ops: none(zero), skip_connect,
                 max_pool_3×3, avg_pool_3×3,
                 sep_conv_3×3, sep_conv_5×5,
                 dil_conv_3×3, dil_conv_5×5
```

### DARTS (Liu et al., ICLR 2019)

Relaxes discrete op choices to continuous weights:

$$\bar{o}^{(i,j)}(x) = \sum_{o \in \mathcal{O}} \underbrace{\frac{e^{\alpha_o^{(i,j)}}}{\sum_{o'} e^{\alpha_{o'}^{(i,j)}}}}_{\text{arch weight}} \cdot o(x)$$

**Bi-level optimization** (alternating):
1. Fix $\alpha$, update $w$ on train split: $\nabla_w \mathcal{L}_{train}(w, \alpha)$
2. Fix $w$, update $\alpha$ on val split: $\nabla_\alpha \mathcal{L}_{val}(w^*(\alpha), \alpha)$

**Discretize**: keep top-2 ops per edge, discard rest.

**Limitations**: memory $O(|\mathcal{O}|)$ during search; performance collapse (skip connections dominate).

### ENAS (Pham et al., ICML 2018)

Weight sharing + RNN controller:
- Controller (LSTM) samples architectures as sequences of decisions
- Child networks share weights from a super-graph
- Train child for 1 mini-batch, compute reward $R$ (validation accuracy)
- Update controller via REINFORCE: $\nabla_\theta J(\theta) = \mathbb{E}_{a \sim \pi_\theta}[R \nabla_\theta \log \pi_\theta(a)]$
- **1000× faster** than original NAS (Zoph & Le, 2017)

### Once-for-All (OFA, Cai et al., ICLR 2020)

Train a **single** OFA network that supports $> 10^{19}$ sub-networks:

```
Progressive Shrinking Training:
  Stage 1: Train full network (max depth, width, kernel, resolution)
  Stage 2: +elastic kernel (7→5→3 with kernel transform)
  Stage 3: +elastic depth  (remove last blocks per unit)
  Stage 4: +elastic width  (sort channels by L1, slice)
  Stage 5: +elastic resolution (interpolate input)

  Deployment: evolutionary search for target latency in ~5 minutes
              No retraining just pick the sub-network!
```

Uses **in-place distillation** at each stage: largest sub-network → smaller sub-networks.

### BigNAS (Yu et al., ECCV 2020)

Train a **big** slimmable network with sandwich sampling:
- **Sandwich rule**: at each step, train max + min + 2 random sub-networks
- **Inplace distillation**: max sub-network acts as teacher for others
- **No post-search retraining** direct deployment
- Supports width 0.25×-1.0×, depth 2-5 blocks, etc.

### Hardware-Aware NAS

Add latency/energy as constraint or objective:
$$\min_\alpha \mathcal{L}_{val}(\alpha) + \lambda \cdot \text{Latency}(\alpha, \text{HW})$$

Latency can be predicted with a lookup table (op × HW → latency) for speed.

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# === DARTS-style mixed operations ===

class MixedOp(nn.Module):
    """DARTS mixed op: differentiable weighted sum of candidate operations."""
    PRIMITIVES = ['skip', 'conv3x3', 'conv1x1', 'avgpool3x3', 'maxpool3x3']

    def __init__(self, C):
        super().__init__()
        self.ops = nn.ModuleList([
            nn.Identity(),
            nn.Sequential(nn.Conv2d(C, C, 3, padding=1, bias=False), nn.BatchNorm2d(C), nn.ReLU()),
            nn.Sequential(nn.Conv2d(C, C, 1, bias=False), nn.BatchNorm2d(C), nn.ReLU()),
            nn.AvgPool2d(3, padding=1, stride=1),
            nn.MaxPool2d(3, padding=1, stride=1),
        ])
        self.arch_params = nn.Parameter(torch.zeros(len(self.ops)))  # start uniform

    def forward(self, x):
        weights = F.softmax(self.arch_params, dim=0)
        return sum(w * op(x) for w, op in zip(weights, self.ops))

    def discretize(self):
        """After search phase: select single best op."""
        best_idx = self.arch_params.argmax().item()
        return self.PRIMITIVES[best_idx], self.ops[best_idx]

class DARTSNode(nn.Module):
    """One node in a DARTS cell (receives 2 inputs)."""
    def __init__(self, C):
        super().__init__()
        self.op0 = MixedOp(C)  # edge from input0
        self.op1 = MixedOp(C)  # edge from input1

    def forward(self, x0, x1):
        return self.op0(x0) + self.op1(x1)

# Demo
torch.manual_seed(42)
C = 32
node = DARTSNode(C)
x = torch.randn(2, C, 8, 8)
out = node(x, x)
print(f'DARTS node output: {out.shape}')

# Architecture params vs weight params
arch_p  = [p for n, p in node.named_parameters() if 'arch' in n]
weight_p = [p for n, p in node.named_parameters() if 'arch' not in n]
print(f'Architecture params: {sum(p.numel() for p in arch_p)} (softmax weights over ops)')
print(f'Weight params:       {sum(p.numel() for p in weight_p)}')

# After some (mock) training what ops were selected?
with torch.no_grad():
    node.op0.arch_params.copy_(torch.tensor([0.1, 2.5, 0.3, 0.2, 0.1]))  # simulate learned
    node.op1.arch_params.copy_(torch.tensor([0.8, 0.2, 0.1, 0.5, 0.1]))

name0, _ = node.op0.discretize()
name1, _ = node.op1.discretize()
print(f'\nDiscretized: edge0 → {name0}, edge1 → {name1}')

# Show op weights
for edge_name, mixed_op in [('edge0', node.op0), ('edge1', node.op1)]:
    weights = F.softmax(mixed_op.arch_params, dim=0)
    print(f'{edge_name}: ' + ', '.join(
        f'{n}={w:.3f}' for n, w in zip(MixedOp.PRIMITIVES, weights)))

# === Once-for-All: elastic width demo ===
print('\n--- Once-for-All: Elastic Width ---')

class ElasticConv(nn.Module):
    """Conv layer that can run at different output widths (OFA-style)."""
    def __init__(self, in_c, max_out_c, kernel=3):
        super().__init__()
        self.full_conv = nn.Conv2d(in_c, max_out_c, kernel, padding=kernel//2, bias=False)
        self.bn = nn.BatchNorm2d(max_out_c)
        self.active_out = max_out_c  # can be reduced at inference

    def forward(self, x):
        w = self.full_conv.weight[:self.active_out]  # slice first N channels
        out = F.conv2d(x, w, padding=self.full_conv.padding[0])
        out = F.relu(self.bn(F.pad(out, [0,0, 0,0, 0, self.full_conv.out_channels - out.size(1)])))
        return out[:, :self.active_out]

ec = ElasticConv(32, 64)
x = torch.randn(2, 32, 16, 16)
for width in [64, 48, 32, 16]:
    ec.active_out = width
    out = ec(x)
    print(f'  Width={width:3d}: output {out.shape}, params used={width*32*3*3}')

DARTS node output: torch.Size([2, 32, 8, 8])
Architecture params: 10 (softmax weights over ops)
Weight params:       20736

Discretized: edge0 → conv3x3, edge1 → skip
edge0: skip=0.065, conv3x3=0.718, conv1x1=0.080, avgpool3x3=0.072, maxpool3x3=0.065
edge1: skip=0.305, conv3x3=0.167, conv1x1=0.151, avgpool3x3=0.226, maxpool3x3=0.151

--- Once-for-All: Elastic Width ---
  Width= 64: output torch.Size([2, 64, 16, 16]), params used=18432
  Width= 48: output torch.Size([2, 48, 16, 16]), params used=13824
  Width= 32: output torch.Size([2, 32, 16, 16]), params used=9216
  Width= 16: output torch.Size([2, 16, 16, 16]), params used=4608


## 7. Efficient Inference Frameworks

### Compilation & Deployment Pipeline

```
PyTorch Model (eager)
      │
      ├──► torch.compile  ──────────► Triton/Inductor kernels (Linux/CUDA)
      │
      ├──► TorchScript (.pt) ─────────► C++ runtime, mobile (LibTorch)
      │
      ├──► ONNX (.onnx) ──────────────► Cross-platform runtime
      │         │
      │         ├──► ONNX Runtime ─────► CPU / CUDA / DirectML
      │         ├──► TensorRT ──────────► NVIDIA GPU (2-8× speedup)
      │         └──► OpenVINO ──────────► Intel CPU/iGPU/VPU
      │
      └──► CoreML (.mlpackage) ────────► Apple CPU/GPU/ANE
```

### Key Optimization Techniques

| Technique | Description | Typical Speedup |
|-----------|-------------|------------------|
| Operator fusion | Merge conv+BN+ReLU into single kernel | 2-5× |
| Kernel tuning | Auto-tune tiling, vectorization | 1.5-3× |
| Memory layout | NCHW→NHWC, channel padding for SIMD | 1.2-2× |
| Constant folding | Pre-compute static expressions | 1.05-1.2× |
| FP16/BF16 casting | Half-precision arithmetic | ~2× |
| INT8 quantization | Integer arithmetic + scale | 3-4× |
| In-place ops | Avoid memory allocation | 10-30% |
| cuDNN autotuner | Best conv algorithm per input shape | 1.2-1.5× |

### torch.compile

PyTorch 2.0+ compilation pipeline: TorchDynamo (bytecode capture) → AOT Autograd → Inductor (Triton codegen).

```python
# Modes:
model = torch.compile(model)                           # default (good balance)
model = torch.compile(model, mode='reduce-overhead')   # fewer Python calls
model = torch.compile(model, mode='max-autotune')      # best perf, slow first run

# Backends:
model = torch.compile(model, backend='inductor')       # default, Triton
model = torch.compile(model, backend='cudagraphs')     # CUDA graph capture
model = torch.compile(model, backend='onnxrt')         # via ONNX Runtime
```

### TensorRT

```
Build phase (offline):
  ONNX → TRT Builder → profile layers → auto-tune kernels → .trt engine

Runtime phase (production):
  Load .trt → ExecutionContext → execute() → output tensors

Optimizations applied:
  - Layer fusion (up to 8+ ops merged)
  - Precision calibration (INT8 with calibration dataset)
  - Kernel selection (timing 100s of implementations)
  - Dynamic shape profiles (min/opt/max shapes)
```

### Flash Attention (Dao et al., 2022)

Standard attention materializes the full $N \times N$ attention matrix:
$$\text{Attn}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Flash Attention** avoids this with tiled computation:

```
Tile Q into blocks Qᵢ, tile K,V into blocks Kⱼ,Vⱼ
For each block:
  1. Compute Sᵢⱼ = QᵢKⱼᵀ/√d  (fits in SRAM)
  2. Update running softmax with online rescaling trick
  3. Accumulate Oᵢ += softmax(Sᵢⱼ)·Vⱼ
  4. Write only Oᵢ to HBM (never write full N×N matrix!)
```

- Memory: $O(N^2) → O(N)$
- Speed: **2-9×** faster than standard attention
- Exact same numerical output (not approximate)
- Flash Attention v2: better parallelism, 2× faster than v1
- Flash Attention v3 (H100): FP8 support, 2.6 PFLOPS

In [8]:
import torch
import torch.nn as nn
import time

# === Efficient Inference Demo ===

class TransformerBlock(nn.Module):
    def __init__(self, dim=256, heads=8):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, heads, batch_first=True, dropout=0.0)
        self.norm2 = nn.LayerNorm(dim)
        self.ff    = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim)
        )

    def forward(self, x):
        h = self.norm1(x)
        h, _ = self.attn(h, h, h, need_weights=False)
        x = x + h
        x = x + self.ff(self.norm2(x))
        return x

model = nn.Sequential(*[TransformerBlock(256) for _ in range(4)])
model.eval()
x = torch.randn(4, 64, 256)

def benchmark(fn, n=30):
    with torch.no_grad():
        for _ in range(3): fn(x)   # warmup
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(n): fn(x)
    return (time.perf_counter() - t0) / n * 1000

# 1. Eager baseline
t_eager = benchmark(model)
print(f'Eager:           {t_eager:.2f} ms')

# 2. TorchScript
try:
    scripted = torch.jit.script(model)
    t_script = benchmark(scripted)
    print(f'TorchScript:     {t_script:.2f} ms  ({t_eager/t_script:.2f}×)')
except Exception as e:
    print(f'TorchScript:     skipped {type(e).__name__}')

# 3. torch.compile (requires PyTorch 2.0+)
try:
    compiled = torch.compile(model, backend='inductor')
    t_compile = benchmark(compiled)
    print(f'torch.compile:   {t_compile:.2f} ms  ({t_eager/t_compile:.2f}×)')
except Exception as e:
    print(f'torch.compile:   not available {type(e).__name__}')

# 4. Dynamic INT8 quantization (linear layers only)
model_dq = torch.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)
t_int8 = benchmark(model_dq)
print(f'Dynamic INT8:    {t_int8:.2f} ms  ({t_eager/t_int8:.2f}×)')

# 5. ONNX export + Runtime
print('\nONNX Export (full workflow):')
ONNX_CODE = '''
import torch.onnx
import onnxruntime as ort
import numpy as np

# Export
torch.onnx.export(
    model, (x,), "model.onnx",
    opset_version=17,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch", 1: "seq_len"}}
)

# Optimize with ORT
opts = ort.SessionOptions()
opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess = ort.InferenceSession("model.onnx", opts,
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"])

# Inference
result = sess.run(None, {"input": x.numpy()})[0]
print(f"ONNX Runtime output: {result.shape}")
'''
print(ONNX_CODE)

# 6. TensorRT workflow
print('TensorRT Workflow (requires tensorrt package):')
TRT_CODE = '''
import tensorrt as trt
import pycuda.driver as cuda

# Build engine from ONNX
logger = trt.Logger(trt.Logger.WARNING)
builder = trt.Builder(logger)
network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
parser = trt.OnnxParser(network, logger)
parser.parse_from_file("model.onnx")

config = builder.create_builder_config()
config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)  # 1GB
# Enable FP16
config.set_flag(trt.BuilderFlag.FP16)

engine = builder.build_serialized_network(network, config)
with open("model.trt", "wb") as f:
    f.write(engine)

# Runtime
runtime = trt.Runtime(logger)
engine = runtime.deserialize_cuda_engine(open("model.trt", "rb").read())
context = engine.create_execution_context()
# ... bind input/output buffers and run
'''
print(TRT_CODE)

Eager:           19.58 ms


TorchScript:     17.48 ms  (1.12×)


torch.compile:   19.77 ms  (0.99×)


/tmp/ipykernel_69492/4276014247.py:59: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_dq = torch.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)


Dynamic INT8:    17.08 ms  (1.15×)

ONNX Export (full workflow):

import torch.onnx
import onnxruntime as ort
import numpy as np

# Export
torch.onnx.export(
    model, (x,), "model.onnx",
    opset_version=17,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch", 1: "seq_len"}}
)

# Optimize with ORT
opts = ort.SessionOptions()
opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess = ort.InferenceSession("model.onnx", opts,
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"])

# Inference
result = sess.run(None, {"input": x.numpy()})[0]
print(f"ONNX Runtime output: {result.shape}")

TensorRT Workflow (requires tensorrt package):

import tensorrt as trt
import pycuda.driver as cuda

# Build engine from ONNX
logger = trt.Logger(trt.Logger.WARNING)
builder = trt.Builder(logger)
network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
parser = trt.OnnxParser(network, logger)
p

## 8. Putting It All Together: Practical Guide

### Method Selection Decision Tree

```
Goal: Deploy model efficiently
│
├─ LLM (7B-70B+ params)?
│   ├─ NVIDIA H100: → FP8 (TransformerEngine / bitsandbytes)
│   ├─ NVIDIA A100/RTX: → AWQ INT4 (best accuracy) or GPTQ INT4
│   ├─ Consumer GPU (24GB): → GGUF Q4_K_M or AWQ
│   ├─ CPU only: → GGUF Q4_K_M (llama.cpp)
│   └─ Fine-tuning on limited GPU: → QLoRA (NF4 + LoRA adapters)
│
├─ CNN / ViT (vision)?
│   ├─ NVIDIA production: → PTQ INT8 + TensorRT
│   ├─ Intel production: → INT8 + OpenVINO
│   ├─ Apple Silicon: → CoreML FP16/INT8
│   ├─ Edge device: → QAT for best accuracy at INT8
│   └─ Mobile: → NAS (OFA/MobileNetV3) + INT8
│
└─ Need smaller model, not just quantized?
    ├─ Same task domain: → Knowledge Distillation
    ├─ Remove redundancy: → Structured pruning + fine-tune
    ├─ LLM pruning: → LLM-Pruner or SliceGPT
    └─ Custom hardware: → NAS with hardware constraints
```

### Compression Pipeline (Best Practice)

```
1. Start with FP16/BF16 baseline (free 2× memory reduction)
2. Apply structured pruning if size budget allows accuracy drop
3. Apply knowledge distillation if training data available
4. Quantize: PTQ first (fast), QAT if accuracy insufficient
5. Compile: ONNX → TensorRT/OpenVINO for production deployment
```

### Bits-per-Weight vs Perplexity (LLaMA-2 7B, WikiText-2)

| Method | Bits/W | PPL | Memory (7B) |
|--------|--------|-----|-------------|
| FP16 baseline | 16.0 | 5.47 | 14 GB |
| GPTQ 4-bit | 4.0 | 5.83 | 3.5 GB |
| AWQ 4-bit | 4.0 | 5.60 | 3.5 GB |
| GGUF Q4_K_M | 4.85 | 5.58 | 4.1 GB |
| GGUF Q8_0 | 8.5 | 5.50 | 7.2 GB |
| SpQR ~4-bit | 4.1 | 5.52 | 3.5 GB |
| HQQ 4-bit | 4.0 | 5.72 | 3.5 GB |
| AQLM 2-bit | 2.0 | 6.95 | 1.75 GB |
| QuIP# 2-bit | 2.0 | 6.84 | 1.75 GB |

### Speedup Summary (A100 GPU, batch=1, 1K token generation)

| Technique | Memory | Throughput | Latency |
|-----------|--------|------------|----------|
| FP16 baseline | 1× | 1× | 1× |
| GPTQ INT4 | 0.25× | 2.5× | 0.4× |
| AWQ INT4 | 0.25× | 3× | 0.35× |
| FP8 (H100) | 0.5× | 4× | 0.25× |
| Flash Attn v2 | 0.95× | 2-4× (long seq) | 0.25× |
| Spec Decoding | 1× | 2-4× | 0.25-0.5× |

In [9]:
# Additional Learning Resources

resources = {
    'Foundational Papers': [
        ('Hinton Knowledge Distillation (2015)',    'https://arxiv.org/abs/1503.02531'),
        ('Lottery Ticket Hypothesis (2019)',         'https://arxiv.org/abs/1803.03635'),
        ('DARTS (2019)',                             'https://arxiv.org/abs/1806.09055'),
        ('ENAS (2018)',                              'https://arxiv.org/abs/1802.03268'),
    ],
    'LLM Quantization Papers': [
        ('GPTQ (2022)',       'https://arxiv.org/abs/2210.17323'),
        ('AWQ (2023)',        'https://arxiv.org/abs/2306.00978'),
        ('SmoothQuant (2022)', 'https://arxiv.org/abs/2211.10438'),
        ('QLoRA (2023)',      'https://arxiv.org/abs/2305.14314'),
        ('SpQR (2023)',       'https://arxiv.org/abs/2306.03078'),
        ('QuIP# (2023)',      'https://arxiv.org/abs/2307.13304'),
        ('HQQ (2023)',        'https://arxiv.org/abs/2309.15531'),
        ('AQLM (2024)',       'https://arxiv.org/abs/2401.06118'),
    ],
    'Pruning Papers': [
        ('SparseGPT (2023)',  'https://arxiv.org/abs/2301.00774'),
        ('Wanda (2023)',      'https://arxiv.org/abs/2306.11695'),
        ('LLM-Pruner (2023)', 'https://arxiv.org/abs/2305.11627'),
        ('SliceGPT (2024)',   'https://arxiv.org/abs/2401.15024'),
    ],
    'Distillation Papers': [
        ('DistilBERT (2019)',       'https://arxiv.org/abs/1910.01108'),
        ('TinyBERT (2019)',         'https://arxiv.org/abs/1909.10351'),
        ('MobileBERT (2020)',       'https://arxiv.org/abs/2004.02984'),
        ('Speculative Decoding (2023)', 'https://arxiv.org/abs/2211.17192'),
    ],
    'NAS Papers': [
        ('Once-for-All (2020)', 'https://arxiv.org/abs/1908.09791'),
        ('BigNAS (2020)',        'https://arxiv.org/abs/2003.11142'),
        ('Flash Attention v2 (2023)', 'https://arxiv.org/abs/2307.08691'),
    ],
    'Libraries & Tools': [
        ('auto-gptq',      'https://github.com/AutoGPTQ/AutoGPTQ'),
        ('autoawq',        'https://github.com/casper-hansen/AutoAWQ'),
        ('bitsandbytes',   'https://github.com/TimDettmers/bitsandbytes'),
        ('llama.cpp',      'https://github.com/ggerganov/llama.cpp'),
        ('ONNX Runtime',   'https://github.com/microsoft/onnxruntime'),
        ('torch-pruning',  'https://github.com/VainF/Torch-Pruning'),
        ('neural-compressor', 'https://github.com/intel/neural-compressor'),
    ],
    'Courses & Surveys': [
        ('MIT 6.5940 TinyML & Efficient Deep Learning', 'https://hanlab.mit.edu/courses/2023-fall-65940'),
        ('HuggingFace Quantization Docs', 'https://huggingface.co/docs/transformers/quantization'),
        ('LLM Quantization Survey (2023)', 'https://arxiv.org/abs/2307.04077'),
        ('Neural Network Pruning Survey (2021)', 'https://arxiv.org/abs/2012.09809'),
    ]
}

print('=' * 60)
print('ADDITIONAL LEARNING RESOURCES')
print('=' * 60)
for category, items in resources.items():
    print(f'\n{category}:')
    for name, url in items:
        print(f'  {name:<40} {url}')

print('\n' + '=' * 60)
print('QUICK INSTALL COMMANDS')
print('=' * 60)
install_cmds = [
    ('GPTQ quantization',      'pip install auto-gptq optimum'),
    ('AWQ quantization',       'pip install autoawq'),
    ('bitsandbytes (QLoRA)',   'pip install bitsandbytes'),
    ('ONNX Runtime GPU',       'pip install onnxruntime-gpu'),
    ('HuggingFace ecosystem',  'pip install transformers accelerate peft'),
    ('llama.cpp Python',       'pip install llama-cpp-python'),
]
for name, cmd in install_cmds:
    print(f'  {name:<30} {cmd}')

ADDITIONAL LEARNING RESOURCES

Foundational Papers:
  Hinton Knowledge Distillation (2015)     https://arxiv.org/abs/1503.02531
  Lottery Ticket Hypothesis (2019)         https://arxiv.org/abs/1803.03635
  DARTS (2019)                             https://arxiv.org/abs/1806.09055
  ENAS (2018)                              https://arxiv.org/abs/1802.03268

LLM Quantization Papers:
  GPTQ (2022)                              https://arxiv.org/abs/2210.17323
  AWQ (2023)                               https://arxiv.org/abs/2306.00978
  SmoothQuant (2022)                       https://arxiv.org/abs/2211.10438
  QLoRA (2023)                             https://arxiv.org/abs/2305.14314
  SpQR (2023)                              https://arxiv.org/abs/2306.03078
  QuIP# (2023)                             https://arxiv.org/abs/2307.13304
  HQQ (2023)                               https://arxiv.org/abs/2309.15531
  AQLM (2024)                              https://arxiv.org/abs/2401.06118

Pruning P